In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import json

In [0]:
df_silver_sales=spark.readStream.format("delta").table("sales")

In [0]:
items_schema = "array<struct<product_id:string,quantity:int,unit_price:double,discount_amount:double,tax_amount:double,total_amount:double>>"

df_silver_sales_1=df_silver_sales.select("created_timestamp","customer_id",explode_outer(from_json(col("items"), items_schema)).alias("product"),"amount","payment_method","sales_id","store_id","transaction_date","flag")


df_silver_sales_selected=df_silver_sales_1.select("*",col("product.product_id").alias("product_id"),col("product.quantity").alias("quantity"),col("product.unit_price").alias("unit_price"),col("product.discount_amount").alias("discount_amount"),col("product.tax_amount").alias("tax_amount"),col("product.total_amount").alias("total_amount")).drop("product")

In [0]:
df_silver_sales_casted=df_silver_sales_selected.withColumns({
    "created_timestamp":col("created_timestamp").cast("timestamp"),
    "updated_timestamp":col("transaction_date").cast("timestamp")
})

In [0]:
df_silver_sales_casted_bad=df_silver_sales_casted.filter(((col("tax_amount") < 1) | (col("customer_id").isNull()) | (col("sales_id").isNull()))).select("*")


df_silver_sales_good=df_silver_sales_casted.filter(((col("tax_amount") >= 1) & (col("customer_id").isNotNull()) & (col("sales_id").isNotNull()))).select("*")

In [0]:
df_silver_sales_good=df_silver_sales_good.withWatermark("updated_timestamp","5 minutes").dropDuplicates(["sales_id","product_id"])


df_silver_sales_store=df_silver_sales_good.groupBy( window("updated_timestamp", "5 minutes"),"store_id").agg(approx_count_distinct("sales_id").alias("sales_count")).select("store_id","sales_count" ,"window")

df_silver_rolling_window=df_silver_sales_good.groupBy( window("updated_timestamp", "5 minutes", "1 minute"),"store_id").agg(approx_count_distinct("sales_id").alias("sales_count")).select("store_id","sales_count" ,"window")






In [0]:
def load_silver_sales(batch_df,batch_id):

    window_con=Window.partitionBy("sales_id","product_id").orderBy(desc("updated_timestamp"))


    batch_df=batch_df.withColumn("rk",row_number().over(window_con)).filter(col("rk")==1).drop("rk")

    batch_df.createOrReplaceTempView("sales_batch")
    
    query= """
    merge into retailsales.silver.sales s using sales_batch t on s.sales_id=t.sales_id and s.product_id=t.product_id when matched and t.updated_timestamp>s.updated_timestamp then update set * when not matched then insert *
    """
    spark.sql(query)

In [0]:
df_silver_sales_good.writeStream.foreachBatch(load_silver_sales).option("checkpointLocation","/Volumes/retailsales/silver/checkpoint/retailsales1").trigger(once=True).start().awaitTermination()

In [0]:
df_silver_rolling_window.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/retailsales/silver/checkpoint/retailsalesagg_rollingwindow").option("mergeSchema","true").trigger(availableNow=True).table("retailsales.silver.store_agg_rolling_window")

In [0]:
df_silver_sales_store.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/retailsales/silver/checkpoint/retailsalesagg_new").option("mergeSchema","true").trigger(availableNow=True).table("retailsales.silver.store_agg")

In [0]:
%sql
select * from retailsales.silver.store_agg_rolling_window

In [0]:
%sql
select * from retailsales.bronze.customers


In [0]:
df_customers=spark.readStream.format("delta").table("retailsales.bronze.customers")
df_customers = df_customers.withColumn(
    "date_of_birth",
    expr("try_cast(date_of_birth as date)")
).withColumn(
    "registration_date",
    expr("try_cast(registration_date as date)")
)

In [0]:
def merge_customers(batch_df,batch_id):
 
    batch_df.createOrReplaceTempView("customers_batch")

    
    query = """
MERGE INTO retailsales.silver.dim_customer t
USING customers_batch s
ON s.customer_id = t.customer_id
AND t.is_current = true

WHEN MATCHED
AND s.created_timestamp > t.effective_from
AND sha1(concat_ws('|',
    s.first_name,
    s.last_name,
    s.gender,
    s.email,
    s.phone_number,
    s.city,
    s.state,
    s.country,
    s.customer_segment,
    cast(s.registration_date as string),
    cast(s.is_active as string)
)) != sha1(concat_ws('|',
    t.first_name,
    t.last_name,
    t.gender,
    t.email,
    t.phone_number,
    t.city,
    t.state,
    t.country,
    t.customer_segment,
    cast(t.registration_date as string),
    cast(t.is_active as string)
))
THEN UPDATE SET
    t.is_current = false,
    t.effective_to = s.created_timestamp
"""

    query1 = """
    MERGE INTO retailsales.silver.dim_customer t
    USING customers_batch s
    ON s.customer_id = t.customer_id
    AND t.is_current = true

    WHEN NOT MATCHED
    THEN INSERT (
        customer_id,
        first_name,
        last_name,
        gender,
        date_of_birth,
        email,
        phone_number,
        city,
        state,
        country,
        customer_segment,
        registration_date,
        is_active,
        created_timestamp,
        effective_from,
        effective_to,
        is_current
    )
    VALUES (
        s.customer_id,
        s.first_name,
        s.last_name,
        s.gender,
        s.date_of_birth,
        s.email,
        s.phone_number,
        s.city,
        s.state,
        s.country,
        s.customer_segment,
        s.registration_date,
        s.is_active,
        s.created_timestamp,
        s.created_timestamp,
        TIMESTAMP '9999-12-31 23:59:59',
        true
    )
    """

    spark.sql(query)
    spark.sql(query1)

In [0]:
df_customers.writeStream.foreachBatch(merge_customers).option("checkpointLocation","/Volumes/retailsales/silver/checkpoint/customers").trigger(availableNow=True).start().awaitTermination()

In [0]:
%sql
select * from retailsales.silver.dim_customer

In [0]:
%sql
select * from retailsales.bronze.products

In [0]:
%sql
CREATE TABLE retailsales.silver.dim_store (
    store_id STRING,
    store_name STRING,
    state STRING,
    city STRING,
    region STRING,
    store_type STRING,
    country STRING,
    open_date DATE,
    manager_id STRING,
    is_active BOOLEAN,
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
df_products=spark.readStream.format("delta").table("retailsales.bronze.products")

In [0]:
def merge_products(batch_df, batch_id):

    # Deduplicate source batch
    window_con = Window.partitionBy("product_id").orderBy(desc("last_updated"))

    batch_df = (
        batch_df
        .withColumn("rk", row_number().over(window_con))
        .filter(col("rk") == 1)
        .drop("rk")
    )

    batch_df.createOrReplaceTempView("products_batch")

    # Step 1: Expire changed current records
    query = """
    MERGE INTO retailsales.silver.dim_product t
    USING products_batch s
    ON t.product_id = s.product_id
       AND t.is_current = true

    WHEN MATCHED
         AND s.last_updated > t.effective_from
         AND (
              NOT (t.sku <=> s.sku)
              OR NOT (t.product_name <=> s.product_name)
              OR NOT (t.category <=> s.category)
              OR NOT (t.subcategory <=> s.subcategory)
              OR NOT (t.brand <=> s.brand)
              OR NOT (t.cost_price <=> s.cost_price)
              OR NOT (t.selling_price <=> s.selling_price)
              OR NOT (t.tax_rate <=> s.tax_rate)
              OR NOT (t.weight_kg <=> s.weight_kg)
              OR NOT (t.is_active <=> s.is_active)
         )
    THEN UPDATE SET
        t.effective_to = s.last_updated,
        t.is_current = false
    """

    spark.sql(query)

    # Step 2: Insert new products + new versions
    query_1 = """
    MERGE INTO retailsales.silver.dim_product t
    USING products_batch s
    ON t.product_id = s.product_id
       AND t.is_current = true

    WHEN NOT MATCHED THEN
    INSERT (
        product_id,
        sku,
        product_name,
        category,
        subcategory,
        brand,
        cost_price,
        selling_price,
        tax_rate,
        weight_kg,
        is_active,
        created_date,
        last_updated,
        effective_from,
        effective_to,
        is_current
    )
    VALUES (
        s.product_id,
        s.sku,
        s.product_name,
        s.category,
        s.subcategory,
        s.brand,
        s.cost_price,
        s.selling_price,
        s.tax_rate,
        s.weight_kg,
        s.is_active,
        s.created_date,
        s.last_updated,
        s.last_updated,
        NULL,
        true
    )
    """

    spark.sql(query_1)

In [0]:
df_products.writeStream \
    .foreachBatch(merge_products) \
    .option(
        "checkpointLocation",
        "/Volumes/retailsales/silver/checkpoint/product_scd2"
    ) \
    .trigger(availableNow=True)\
    .start() \
    .awaitTermination()

In [0]:
%sql
select * from retailsales.bronze.stores

In [0]:
df_stores=spark.readStream.format("delta").table("retailsales.bronze.stores")

df_stores=df_stores.withColumns({
    "state":get_json_object(col("location"),"$.state"),
    "city":get_json_object(col("location"),"$.city"),
    "manager_id":get_json_object(col("manager"),"$.ManagerID")
}).drop("location","manager")

In [0]:
def merge_stores(batch_df, batch_id):

    batch_df = (
        batch_df
        .withColumn("is_active", col("IsActive").cast("boolean"))
        .withColumn("open_date", to_date(col("OpenDate")))
        .withColumn("country", col("Country"))
        .withColumn("effective_from", current_timestamp())
    )

    # One record per store in this micro-batch
    window_con = Window.partitionBy("store_id").orderBy(
        desc("effective_from")
    )

    batch_df = (
        batch_df
        .withColumn("rk", row_number().over(window_con))
        .filter(col("rk") == 1)
        .drop("rk")
    )

    batch_df.createOrReplaceTempView("stores_batch")

    # Step 1: Expire changed current records
    query = """
    MERGE INTO retailsales.silver.dim_store t
    USING stores_batch s
    ON t.store_id = s.store_id
       AND t.is_current = true

    WHEN MATCHED
         AND (
              NOT (t.store_name <=> s.store_name)
              OR NOT (t.state <=> s.state)
              OR NOT (t.city <=> s.city)
              OR NOT (t.region <=> s.region)
              OR NOT (t.store_type <=> s.store_type)
              OR NOT (t.country <=> s.country)
              OR NOT (t.open_date <=> s.open_date)
              OR NOT (t.manager_id <=> s.manager_id)
              OR NOT (t.is_active <=> s.is_active)
         )
    THEN UPDATE SET
        t.effective_to = s.effective_from,
        t.is_current = false
    """

    spark.sql(query)

    # Step 2: Insert new stores / new versions
    query_1 = """
    MERGE INTO retailsales.silver.dim_store t
    USING stores_batch s
    ON t.store_id = s.store_id
       AND t.is_current = true

    WHEN NOT MATCHED THEN
    INSERT (
        store_id,
        store_name,
        state,
        city,
        region,
        store_type,
        country,
        open_date,
        manager_id,
        is_active,
        effective_from,
        effective_to,
        is_current
    )
    VALUES (
        s.store_id,
        s.store_name,
        s.state,
        s.city,
        s.region,
        s.store_type,
        s.country,
        s.open_date,
        s.manager_id,
        s.is_active,
        s.effective_from,
        NULL,
        true
    )
    """

    spark.sql(query_1)

In [0]:
df_stores.writeStream \
    .foreachBatch(merge_stores) \
    .option(
        "checkpointLocation",
        "/Volumes/retailsales/silver/checkpoint/store_scd2"
    ) \
    .trigger(availableNow=True)\
    .start() \
    .awaitTermination()